# Tracing a multi-step app — where tracing earns its keep

*Part of the `gen_ai/` track. Prerequisite: `a_tracing_quickstart`.*

## The problem: one bad answer, many possible culprits

`a_tracing_quickstart` traced a single LLM call. Real GenAI apps are **pipelines**: retrieve some context, build a prompt, call the model, maybe parse or call a tool. When the final answer is wrong, *which step* failed?

- Did **retrieval** fetch the wrong documents?
- Did the **prompt** drop the good context?
- Did the **model** ignore what it was given?

A flat log can't tell you. A **trace** can: each step is a **span**, nested under the request, so you read the failure top-to-bottom. We'll build a tiny **RAG** (retrieval-augmented generation) app — by hand, no framework — so the spans are completely transparent.

## Prerequisites

Same as `a_tracing_quickstart` — nothing new:

- A tracking server on port 5001 (`mlflow server --host 127.0.0.1 --port 5001` from `src/`).
- Ollama serving `qwen3:8b` (or set `MODEL` to any chat model you pulled).

No vector database, no LangChain — a few in-memory documents and a one-line retriever keep the focus on *tracing*, not retrieval quality.

In [ ]:
import mlflow
from mlflow.entities import SpanType
from openai import OpenAI

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("genai-multistep-app")
mlflow.openai.autolog()  # traces the LLM call inside the pipeline

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
MODEL = "qwen3:8b"

## A tiny "knowledge base"

Four one-line documents about MLflow. In a real app these would come from a vector store; here they're a Python list so nothing is hidden.

In [ ]:
DOCS = [
    {"id": "tracking", "text": "MLflow Tracking logs parameters, metrics, and artifacts for each run."},
    {"id": "registry", "text": "The MLflow Model Registry version-controls models and assigns aliases like champion."},
    {"id": "serving",  "text": "mlflow models serve starts a REST endpoint that answers /invocations requests."},
    {"id": "tracing",  "text": "MLflow Tracing records spans for each step of a GenAI app so you can debug it."},
]

## Step 1 — a traced retriever

`@mlflow.trace(span_type=SpanType.RETRIEVER)` marks this as a **retrieval** span — MLflow tags it specially in the UI (and retrieval-specific scorers in `c_genai_evaluation` can grade it). The retriever just scores documents by word overlap with the question and returns the top `k`.

In [ ]:
@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve(question: str, k: int = 2) -> list[dict]:
    q = set(question.lower().replace("?", "").split())
    ranked = sorted(DOCS, key=lambda d: len(q & set(d["text"].lower().split())), reverse=True)
    return ranked[:k]

retrieve("How do I serve a model over REST?")  # quick look at what it returns

## Step 2 — the RAG pipeline

`answer()` ties it together: retrieve context, stuff it into the prompt, call the model. `@mlflow.trace(span_type=SpanType.CHAIN)` makes it the **root** span; the `retrieve` call and the autologged LLM call nest underneath it as children.

In [ ]:
@mlflow.trace(span_type=SpanType.CHAIN)
def answer(question: str) -> str:
    docs = retrieve(question)
    context = "\n".join(f"- {d['text']}" for d in docs)
    messages = [
        {"role": "system",
         "content": "Answer using ONLY the context. If the answer isn't in it, say you don't know."},
        {"role": "user",
         "content": f"Context:\n{context}\n\nQuestion: {question} /no_think"},
    ]
    resp = client.chat.completions.create(model=MODEL, messages=messages)
    return resp.choices[0].message.content.strip()

print(answer("How do I serve a model over REST?"))

## Read the trace

Open <http://127.0.0.1:5001> → **Traces**. The newest trace is a tree:

```
answer                      (CHAIN)   — the whole request
├─ retrieve                 (RETRIEVER) — inputs: the question; outputs: the 2 docs it picked
└─ Completions.create       (LLM)     — the prompt (with context) and the model's reply
```

You can now see, for one request, **exactly what each step received and produced** — the docs the retriever chose and the prompt the model actually saw.

## Why this matters: diagnosing a bad answer

Ask something the knowledge base doesn't cover and watch the trace explain itself:

In [ ]:
print(answer("What's the weather in Athens today?"))

The model should answer something like *"I don't know."* — and the **trace tells you why**: the `retrieve` span returned MLflow docs with near-zero overlap, so there was no useful context to answer from. The problem is **retrieval**, not the prompt or the model.

That's the payoff. Without the trace you'd see only the final "I don't know" and have to guess. With it, the failing step is right there — which is the whole reason production GenAI systems are instrumented.

**Frameworks do this for you.** LangChain, LlamaIndex, DSPy and others have one-line `mlflow.<framework>.autolog()` that traces every retriever / chain / tool step automatically — the same spans you built by hand here. We built it manually so you can see exactly what a span *is* before letting a framework generate them.

## Next steps

- **`c_genai_evaluation.ipynb`** — now that the app produces traces, *score* it: `mlflow.genai.evaluate()` with LLM-as-judge, including retrieval scorers (`RetrievalRelevance`, `RetrievalGroundedness`) that grade the very `RETRIEVER` span you just created.